# RENER

Idea:
1. Import a portion of the labeled dataset into the RAG DB
2. Given the input, query the DB for the most similar sentence (BM25 or semantic) 
3. Output the most similar result(s) in an "Input: \<input\>\nOutput: \<output\>" format
The idea is to use the results for few-shot prompting, using the most similar resutls for each input

In [ ]:
import json
from sklearn.model_selection import train_test_split

# Load jsonl file into a list of dictionaries
with open("../data/labelled/all_parallel.jsonl") as test_f:
    data = [json.loads(line) for line in test_f]

# Create test split (notice: fixed seed)
train_data, test_data = train_test_split(data, test_size=0.3, random_state=52)

In [ ]:
from haystack import Pipeline
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.dataclasses import Document
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever

# 1. Create document store
document_store = InMemoryDocumentStore()

# 2. Write documents into the store
docs = [Document(content=d["input"], meta={"output": d["text"]}) for d in train_data]
document_store.write_documents(docs, policy="overwrite")

# 3. Create BM25 retriever
retriever = InMemoryBM25Retriever(document_store=document_store)

# 4. Build pipeline
pipeline = Pipeline()
pipeline.add_component(name="retriever", instance=retriever)

# 5. Run the pipeline
result = pipeline.run(
    data={
        "retriever": {
            "query": "Il Riccardo Galli ha specificato di aver conosciuto il Paride Valli presso la macelleria, sita nel quartiere Salamini, dei propri cugini Ettore Vitale, Ottavio Neri ed Leandro Sala (l’altro collaborante già citato), frequentata dal padre Riccardo Galli e da altri uomini d’altri tempi (come Gambino Simone Costa) e che il Paride Valli era persona vicina ed amica di Tullio Conte Di Napoli, circostanza anche personalmente constatata dal collaboratore allorché, in alcune occasioni, aveva incontrato il Paride Valli in compagnia del Di Napoli nella villa di proprietà di Enrico Fontana Gino Basile, altro uomo d’altri tempi della famiglia di Orlando Pace, nonché parente dello stesso Paride Valli ove lo stesso Di Napoli dimorava durante la sua gita.",
            "top_k": 3,
        }
    }
)

# 6. Display results
for doc in result["retriever"]["documents"]:
    print(
        f"Score: {doc.score:.2f} | Content: {doc.content} | Output: {doc.meta['output']}"
    )


In [ ]:
from haystack import Document, Pipeline
from haystack.components.builders.prompt_builder import PromptBuilder
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever
from haystack.document_stores.in_memory import InMemoryDocumentStore

from haystack_integrations.components.generators.ollama import OllamaGenerator

document_store = InMemoryDocumentStore()
document_store.write_documents(
    [
        Document(content="Super Mario was an important politician"),
        Document(
            content="Mario owns several castles and uses them to conduct important political business"
        ),
        Document(
            content="Super Mario was a successful military leader who fought off several invasion attempts by "
            "his arch rival - Bowser"
        ),
    ]
)

template = """
Given only the following information, answer the question.
Ignore your own knowledge.

Context:
{% for document in documents %}
    {{ document.content }}
{% endfor %}

Question: {{ query }}?
"""

pipe = Pipeline()

pipe.add_component("retriever", InMemoryBM25Retriever(document_store=document_store))
pipe.add_component("prompt_builder", PromptBuilder(template=template))
pipe.add_component(
    "llm",
    OllamaGenerator(
        model="llama3.1:8b",
        # model="gemma3:4b",
        url="http://localhost:11434",
    ),
)
pipe.connect("retriever", "prompt_builder.documents")
pipe.connect("prompt_builder", "llm")

query = "Who is Super Mario?"

response = pipe.run({"prompt_builder": {"query": query}, "retriever": {"query": query}})

print(response["llm"]["replies"])

In [ ]:
from haystack import Pipeline
from haystack.dataclasses import Document
from haystack.components.writers import DocumentWriter

# Note: the following requires a "pip install sentence-transformers"
from haystack.components.embedders import (
    SentenceTransformersDocumentEmbedder,
    SentenceTransformersTextEmbedder,
)

from haystack_integrations.document_stores.chroma import ChromaDocumentStore
from haystack_integrations.components.retrievers.chroma import ChromaEmbeddingRetriever
from sentence_transformers import SentenceTransformer

# Chroma is used in-memory so we use the same instances in the two pipelines below
document_store = ChromaDocumentStore()

documents = [
    Document(content="Super Mario was an important politician"),
    Document(
        content="Mario owns several castles and uses them to conduct important political business"
    ),
    Document(
        content="Super Mario was a successful military leader who fought off several invasion attempts by "
        "his arch rival - Bowser"
    ),
]

indexing = Pipeline()
indexing.add_component(
    "embedder",
    SentenceTransformersDocumentEmbedder(
        model="sentence-transformers/all-MiniLM-L6-v2",
    ),
)
indexing.add_component("writer", DocumentWriter(document_store))
indexing.connect("embedder.documents", "writer.documents")
indexing.run({"embedder": {"documents": documents}})

querying = Pipeline()
querying.add_component(
    "query_embedder",
    SentenceTransformersTextEmbedder(
        model="sentence-transformers/all-MiniLM-L6-v2",
        token=False,
    ),
)
querying.add_component("retriever", ChromaEmbeddingRetriever(document_store))
querying.connect("query_embedder.embedding", "retriever.query_embedding")

results = querying.run({"query_embedder": {"text": "Who is Super Mario?"}})

for d in results["retriever"]["documents"]:
    print(d.meta, d.score)